# Lab 04a: O Peso da Precisão (Quantização)

**Disciplina:** Sistemas Distribuídos para IA (1CXIA - S10)
**Objetivo:** Medir o impacto da precisão numérica (FP32 vs. INT4) na performance de modelos de linguagem (LLMs).
**Ambiente:** Google Colab (Exclusivo).

## 🚀 Instruções de Acesso ao Google Colab

Este laboratório requer uma GPU para funcionar, devido à biblioteca `bitsandbytes`. Siga os passos abaixo:

1. Acesse: [https://colab.research.google.com/](https://colab.research.google.com/) e crie um **Novo Notebook**.
2. **ATIVAR GPU:** Vá em `Ambiente de Execução` -> `Alterar o tipo de ambiente de execução`.
3. Selecione **T4 GPU** e clique em **Salvar**.
4. Confirme se o ícone verde no canto superior direito mostra **"T4"**.

## 1. Setup do Ambiente
Execute a célula abaixo para instalar as bibliotecas de otimização:

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate

## 2. Passo 1: O Modelo em Precisão Total (Baseline)
Vamos carregar o modelo `TinyLlama-1.1B` em sua forma original. Mesmo sendo um modelo pequeno, veremos o consumo de memória.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Carregando Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Carregando Modelo em FP32 (Original)
start_time = time.time()
model_fp32 = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float32, 
    device_map="auto"
)
load_time_fp32 = time.time() - start_time

print(f"✅ Modelo FP32 carregado em {load_time_fp32:.2f} segundos.")
print(f"💾 Memória Ocupada (VRAM): {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

## 3. Passo 2: O Benchmark de Velocidade
Vamos gerar uma resposta e medir quantos **tokens por segundo** o sistema consegue entregar.

In [ ]:
messages = [{"role": "user", "content": "Explique o que é um sistema distribuído em 3 frases."}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

# Benchmark FP32
start_gen = time.time()
output = model_fp32.generate(
    **input_ids, 
    max_new_tokens=100, 
    do_sample=True, 
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)
end_gen = time.time()

# Calculando apenas os novos tokens gerados
new_tokens = output[0][input_ids.input_ids.shape[-1]:]
tps_fp32 = len(new_tokens) / (end_gen - start_gen)

print(f"🚀 Performance FP32: {tps_fp32:.2f} tokens/seg (Geração)")
print(f"📝 Resposta: {tokenizer.decode(new_tokens, skip_special_tokens=True)}")

# Limpando memória para o próximo passo
del model_fp32
torch.cuda.empty_cache()

## 4. Passo 3: A Dieta (Quantização 4-bit)
Agora, vamos carregar o **mesmo modelo**, mas forçando cada parâmetro a ocupar apenas **4 bits** usando a tecnologia `bitsandbytes`.

In [ ]:
from transformers import BitsAndBytesConfig

# Configuração de Quantização 4-bit (NF4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Carregando Modelo Quantizado
start_time = time.time()
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config,
    device_map="auto"
)
load_time_4bit = time.time() - start_time

print(f"✅ Modelo 4-bit carregado em {load_time_4bit:.2f} segundos.")
print(f"💾 Memória Ocupada (VRAM): {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

## 5. Passo 4: O Benchmark Final (Comparação)
Repita o teste de geração e observe o ganho de Throughput.

In [ ]:
# Benchmark 4-bit
start_gen = time.time()
output = model_4bit.generate(
    **input_ids, 
    max_new_tokens=100, 
    do_sample=True, 
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id
)
end_gen = time.time()

# Calculando apenas os novos tokens gerados
new_tokens_4b = output[0][input_ids.input_ids.shape[-1]:]
tps_4bit = len(new_tokens_4b) / (end_gen - start_gen)

print(f"🔥 Performance 4-bit: {tps_4bit:.2f} tokens/seg (Geração)")
print(f"📈 Ganho de Velocidade: {tps_4bit / tps_fp32:.2f}x")
print(f"📝 Resposta: {tokenizer.decode(new_tokens_4b, skip_special_tokens=True)}")